<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/01_the_agent_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — The agent loop, from scratch

**The claim you should be able to make when you finish:** *"An agent is a loop
over a model, its tools and a transcript. I've written the twenty lines, I know
the four invariants that fail silently, and I can read any framework as a set of
choices about those three things."*

"Agent" gets used to mean a framework, a product category and a vibe. None of
those are things you can debug. This is:

```python
while True:
    response = model(system, messages, tools)
    if response.stop_reason != "tool_use":
        break
    messages.append(assistant(response.content))
    messages.append(user([execute(call) for call in response.tool_calls]))
```

Everything else — memory, planning, multi-agent, skills, guardrails — is a
choice about **what goes in `system`**, **what goes in `tools`**, and **what you
do to `messages` between iterations**. Once you have written this yourself,
every framework becomes readable, because you can see which of those three knobs
it is turning and what it charges you for the privilege.

Forty minutes. No GPU, no API key, no spend — the model here is a fake one, on
purpose, and section 6 explains why that is the right call and not a compromise.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 1. A tool is two things

A schema the model sees, and a function your harness runs. Keeping those two
separate is the whole design.

Note what is *not* in the schema: `read_only` and `destructive`. The model does
not need to know, and you would not want to trust it with the knowledge. Those
flags are for the harness — what to gate behind a confirmation, what is safe to
run in parallel, what to log loudly.

That split is the entire argument for promoting an action out of a
general-purpose `bash` tool. A typed `send_email(to, body)` can be gated,
rendered and audited. `bash("curl -X POST ...")` is an opaque string, and the
harness cannot tell it from `ls`.

In [ ]:
from agentlab.loop import Tool, ToolRegistry

STOCK = {"widget": 12, "sprocket": 7, "gizmo": 30}

lookup = Tool(
    name="lookup_stock",
    description="Return the number of units of one product currently in stock.",
    input_schema={
        "type": "object",
        "properties": {"product": {"type": "string", "description": "Product name."}},
        "required": ["product"],
    },
    fn=lambda product: STOCK[product],
)

restock = Tool(
    name="restock",
    description="Order more units of a product. Cannot be undone once submitted.",
    input_schema={
        "type": "object",
        "properties": {"product": {"type": "string"}, "units": {"type": "integer"}},
        "required": ["product", "units"],
    },
    fn=lambda product, units: f"ordered {units} {product}",
    read_only=False,
    destructive=True,      # the harness knows; the model does not need to
)

tools = ToolRegistry([lookup, restock])
print("what the model sees:")
for schema in tools.schemas():
    print(" ", schema["name"], "->", sorted(schema["input_schema"].get("properties", {})))
print("\nwhat the harness knows:", {t.name: {"destructive": t.destructive} for t in tools})

## 2. The transcript is the state

The model is **stateless**. It does not remember your last turn; you resend it.
That single fact has two consequences, and they are the two halves of this whole
repo:

> Anything not in `messages` did not happen.
>
> Everything in `messages` is being paid for again, on every single turn.

The first drives everything in lab 7 (memory). The second drives everything in
lab 2 (cost). Hold onto both.

Here is what a transcript actually looks like — plain dicts, exactly the shapes
that go on the wire. No wrapper classes, deliberately: the shapes *are* the API,
and a wrapper is one more thing to be wrong about.

In [ ]:
from agentlab.loop import assistant, text_block, tool_result_block, tool_use_block, user

transcript = [
    user("How many widgets do we have?"),
    assistant([tool_use_block(id="t1", name="lookup_stock", input={"product": "widget"})]),
    user([tool_result_block(tool_use_id="t1", content="12")]),
    assistant("There are 12 widgets in stock."),
]

for msg in transcript:
    kinds = [b["type"] for b in msg["content"]]
    print(f"{msg['role']:<10} {kinds}")

## 3. A model you can reason about

The lessons in this repo run against fake models. That is not a shortcut around
the interesting part — it *is* the interesting part.

The loop, the stop conditions, the error handling, the budget caps and the
context management are all **your code**. Testing your code against a paid,
slow, non-deterministic API is slower, costs money, and tells you less, because
when something breaks you cannot tell whether it was you or the sampler.

`ScriptedModel` replays a fixed list of responses. It is exactly the right tool
for asking "does my harness do the right thing when...".

In [ ]:
from agentlab.loop import ModelResponse, ScriptedModel, run

model = ScriptedModel([
    ModelResponse([tool_use_block("t1", "lookup_stock", {"product": "widget"})], "tool_use"),
    ModelResponse([text_block("There are 12 widgets in stock.")], "end_turn"),
])

trace = run(model, tools, "How many widgets do we have?")
print(trace)
print()
for call in trace.calls:
    print(f"  {call.name}({call.input}) -> {call.result}")

## 4. Four invariants, all of which fail silently

This is the part worth slowing down for. None of these raise an exception when
you get them wrong. They produce a system that is subtly worse, and you find out
weeks later from a chart.

### Invariant 1 — every `tool_use` gets exactly one `tool_result`

Keyed by id. Not optional, not reorderable, not skippable — **including for the
call that failed**. Drop one and the API rejects the transcript, usually three
turns after the mistake.

### Invariant 2 — a tool failure is a `tool_result` with `is_error`, not an exception

An exception kills the run. An error message is a chance for the model to
recover — which makes it a **prompt**, and the only one in the loop written
specifically for a model that has just made a mistake and is deciding what to do
next. Lab 4 is largely about this.

### Invariant 3 — all results from one assistant turn go back in *one* user message

This is the quiet one. Splitting parallel results across several messages does
not error. It teaches the model, by example, that parallel calls are not how
this conversation works — and it stops making them. You lose the parallelism and
nothing anywhere says so.

### Invariant 4 — the transcript is append-only within a turn

You may rewrite history *between* turns (that is what compaction is), but not
midway through one.

Let's break each of them on purpose.

In [ ]:
from agentlab.loop import LoopInvariantError, check_transcript

def try_it(label, messages):
    try:
        check_transcript(messages)
        print(f"ok    {label}")
    except LoopInvariantError as exc:
        print(f"CAUGHT {label}\n       {exc}")

# 1. A tool_use nobody answered.
try_it("unanswered tool_use", [
    user("hi"), assistant([tool_use_block("a", "lookup_stock", {})]),
])

# 3. Two parallel calls, results split across two user messages.
try_it("parallel results split in two", [
    user("hi"),
    assistant([tool_use_block("a", "lookup_stock", {}), tool_use_block("b", "lookup_stock", {})]),
    user([tool_result_block("a", "12")]),
    user([tool_result_block("b", "7")]),
])

# A result for a call that never happened.
try_it("result for a phantom call", [
    user("hi"), assistant([tool_use_block("a", "lookup_stock", {})]),
    user([tool_result_block("ghost", "12")]),
])

try_it("a well-formed transcript", [
    user("hi"), assistant([tool_use_block("a", "lookup_stock", {})]),
    user([tool_result_block("a", "12")]), assistant("There are 12."),
])

`check_transcript` is worth stealing into your own harness's test suite. It costs
nothing to run and it catches the class of bug that is otherwise diagnosed from a
latency dashboard.

Now watch the loop get it right, with two calls in one turn.

In [ ]:
model = ScriptedModel([
    ModelResponse([
        tool_use_block("t1", "lookup_stock", {"product": "widget"}),
        tool_use_block("t2", "lookup_stock", {"product": "gizmo"}),
    ], "tool_use"),
    ModelResponse([text_block("12 widgets and 30 gizmos.")], "end_turn"),
])

trace = run(model, tools, "widgets and gizmos?")
for msg in trace.meta["messages"]:
    kinds = [b["type"] for b in msg["content"]]
    print(f"{msg['role']:<10} {len(kinds)} block(s): {kinds}")

print("\nBoth results, one user message. That is invariant 3, held.")

## 5. Failure is a message, not a crash

Two failures every real agent meets on day one: a tool that raises, and a tool
name the model invented. Neither should end the run.

Watch what comes back for the hallucinated name. It does not just say "no". It
says which names *do* exist — turning a dead end into a recoverable step. That
is thirty tokens buying a turn, and turns are the expensive unit.

In [ ]:
exploding = Tool("check_price", "Return the current price of a product.",
                 fn=lambda product: 1 / 0)
registry = ToolRegistry([lookup, exploding])

model = ScriptedModel([
    ModelResponse([tool_use_block("t1", "check_price", {"product": "widget"})], "tool_use"),
    ModelResponse([tool_use_block("t2", "lookup_prices", {})], "tool_use"),   # invented
    ModelResponse([text_block("I could not get prices; here is stock instead.")], "end_turn"),
])

trace = run(model, registry, "what do widgets cost?")
for call in trace.calls:
    print(f"  {call.name:<14} error={call.is_error}  -> {call.result}")
print(f"\nThe run survived both and finished on {trace.stop_reason!r}.")

## 6. Stop conditions — all three of them fire in production

A loop with only "the model stopped asking for tools" is a runaway waiting for a
bad afternoon. There are three, and they are checked in this order:

1. **the model stopped asking** — `stop_reason != "tool_use"`. This is success.
2. **step limit** — the circuit breaker. Catches loops.
3. **token budget** — the money circuit breaker. Catches expensive loops, which
   are not the same set as long ones: one tool returning 200K tokens can cost
   more than fifty cheap turns.

Here is a model that will never stop on its own.

In [ ]:
def stubborn(messages, tools):
    return ModelResponse([tool_use_block(f"t{len(messages)}", "lookup_stock",
                                         {"product": "widget"})], "tool_use")

from agentlab.loop import PolicyModel

capped = run(PolicyModel(stubborn), tools, "go", max_steps=8)
print("step limit:  ", capped.stop_reason, "after", capped.steps, "steps")

broke = run(PolicyModel(stubborn), tools, "go", max_steps=1000, max_tokens_budget=5_000)
print("token budget:", broke.stop_reason, "after", broke.steps, "steps")

Note the second run: `max_steps=1000` would not have saved you. The budget did.
Ship both.

## 7. The trajectory is the second output

Every run produces two things. The **answer**, which everyone looks at, and the
**trajectory** — the calls, the errors, the repeats — which is the one that tells
you whether the answer was luck.

Two runs can both succeed with wildly different trajectories: one tool call, or
six retries of a call that was wrong the first time and wrong the same way five
more times. Outcome-only evals score those identically. That is how a system
holds its pass rate for a quarter while its cost per task triples.

The cheapest derailment signal there is needs no judge, no labels and no model:
**the same tool, called with the same arguments, more than once.**

In [ ]:
def spinning():
    """A fresh model each time — a ScriptedModel is consumed as it is read."""
    return ScriptedModel([
        ModelResponse([tool_use_block(f"t{i}", "lookup_stock", {"product": "widget"})], "tool_use")
        for i in range(6)
    ] + [ModelResponse([text_block("12")], "end_turn")])

trace = run(spinning(), tools, "how many widgets?")
print(f"tool calls:     {trace.tool_calls}")
print(f"distinct calls: {trace.distinct_calls}")
print(f"repeat calls:   {trace.repeat_calls}   <- free derailment signal")

alternating = ScriptedModel([
    ModelResponse([tool_use_block(f"t{i}", "lookup_stock",
                                  {"product": "widget" if i % 2 else "gizmo"})], "tool_use")
    for i in range(6)
] + [ModelResponse([text_block("done")], "end_turn")])
print(f"\nA-B-A-B cycle length: {run(alternating, tools, 'go').longest_cycle()}")

## 8. The hook where everything else lives

`run(..., on_step=...)` fires between iterations, with the turn and the mutable
transcript. That one hook is where **every** technique in the rest of this repo
lives:

| What you do in `on_step` | Which lab |
|---|---|
| drop old tool results | 7 — memory |
| summarise the transcript when it gets long | 7 — memory |
| break the loop when calls repeat | 5 — the doom loop |
| ask a human before a destructive call | 9 — security |
| count tokens and stop | 2 — cost |

Here is the loop-breaker, in five lines.

In [ ]:
seen = set()

def break_the_loop(turn, messages):
    for call in turn.tool_calls:
        if call.key in seen:
            messages.append(user(
                "You have already made that exact call and received that exact result. "
                "Use what you have, or try a different approach."))
            return
        seen.add(call.key)

trace = run(spinning(), tools, "how many widgets?", on_step=break_the_loop)
print("nudges injected:",
      sum(1 for m in trace.meta["messages"]
          if m["role"] == "user" and m["content"][0].get("text", "").startswith("You have already")))
print("Five lines in the harness. No prompt engineering, no model change.")

## What you can now say

- *"An agent is a loop over model, tools and a transcript — here are the twenty
  lines."*
- *"The transcript is the state. Anything not in it didn't happen, and
  everything in it is billed again every turn."*
- *"Parallel tool results go back in one user message. Split them and the model
  quietly stops calling tools in parallel."*
- *"A tool failure is a `tool_result`, not an exception — and the error text is a
  prompt for a model that just made a mistake."*
- *"Three stop conditions, and the token budget catches runs the step limit
  won't."*
- *"Repeat calls with identical arguments are the cheapest derailment signal
  there is."*

## Next

You now know that the transcript is resent on every turn. **[Lab 2](02_context_economics.ipynb)
works out what that costs**, and the answer is not linear.